# Experimento: CNN sobre CIFAR-10

**Código de matrícula:** 019100568J · **Semilla:** 68

Notebook de pruebas: carga del dataset con semilla, EDA, visualización de muestras, Data Augmentation,
partición estratificada y validación de shapes del modelo. El entrenamiento completo está en `src/train.py`.

## 1. Configuración y semilla

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))   # el notebook vive en notebooks/

import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

from dataset import (SEED, CLASSES, DATA_DIR, CIFAR10_MEAN, CIFAR10_STD,
                     set_seed, get_transforms, load_cifar10, stratified_split,
                     get_dataloaders, show_samples)
from model import CNN_CIFAR10, validate_shapes, count_parameters

set_seed(SEED)                 # torch.manual_seed(68), np.random.seed(68), ...
print("Semilla:", SEED)
print("PyTorch:", torch.__version__)

## 2. Carga de CIFAR-10

In [ ]:
transform_train, transform_eval = get_transforms()
train_aug, train_eval, test_set = load_cifar10(transform_train, transform_eval, download=True)

print("Entrenamiento:", len(train_aug), "imágenes")
print("Prueba       :", len(test_set), "imágenes")
print("Clases       :", CLASSES)

img, label = train_eval[0]
print("Shape de una imagen transformada:", tuple(img.shape), "-> (Canales, Alto, Ancho)")
print("Tipo:", img.dtype, "| rango tras normalizar: [%.2f, %.2f]" % (img.min(), img.max()))

## 3. Análisis exploratorio (EDA)

In [ ]:
train_targets = np.array(train_aug.targets)
test_targets = np.array(test_set.targets)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(10)
ax.bar(x - 0.2, np.bincount(train_targets), width=0.4, label="Entrenamiento")
ax.bar(x + 0.2, np.bincount(test_targets), width=0.4, label="Prueba")
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=30)
ax.set_ylabel("Número de imágenes")
ax.set_title("Distribución de clases en CIFAR-10")
ax.legend()
plt.show()

print("Imágenes por clase (entrenamiento):", np.bincount(train_targets).tolist())
print("Imágenes por clase (prueba)       :", np.bincount(test_targets).tolist())

In [ ]:
# Estadísticos por canal RGB (sobre los datos crudos en [0, 1])
raw = datasets.CIFAR10(DATA_DIR, train=True, download=False)
data = raw.data / 255.0                     # numpy (50000, 32, 32, 3)
print("Shape del arreglo crudo:", data.shape, "-> (N, Alto, Ancho, Canales)")
print("Media por canal (R, G, B):", data.mean(axis=(0, 1, 2)).round(4))
print("Desv. estándar  (R, G, B):", data.std(axis=(0, 1, 2)).round(4))
print("Valores usados en Normalize:", CIFAR10_MEAN, CIFAR10_STD)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, (c, name) in enumerate(zip(["red", "green", "blue"], ["R", "G", "B"])):
    axes[i].hist(data[:5000, :, :, i].ravel(), bins=50, color=c, alpha=0.7)
    axes[i].set_title(f"Histograma canal {name} (5000 imágenes)")
plt.tight_layout()
plt.show()

## 4. Visualización de 16 imágenes con su etiqueta real

In [ ]:
show_samples(n=16, seed=SEED, save_path=Path.cwd().parent / "outputs" / "samples_grid.png")

## 5. Data Augmentation
La misma imagen pasada 6 veces por `RandomCrop` + `RandomHorizontalFlip`.

In [ ]:
aug_only = transforms.Compose([transforms.RandomCrop(32, padding=4),
                               transforms.RandomHorizontalFlip()])
img, label = raw[7]
fig, axes = plt.subplots(1, 7, figsize=(14, 2.5))
axes[0].imshow(img); axes[0].set_title("Original"); axes[0].axis("off")
for ax in axes[1:]:
    ax.imshow(aug_only(img)); ax.set_title("Aumentada"); ax.axis("off")
fig.suptitle(f"Clase: {CLASSES[label]}")
plt.show()

## 6. Partición estratificada entrenamiento / validación

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(batch_size=128, val_size=0.1, seed=SEED)

## 7. Modelo y validación de shapes 4D

In [ ]:
torch.manual_seed(SEED)
model = CNN_CIFAR10()
print(model)
validate_shapes(model, batch_size=8)

In [ ]:
# Prueba con un batch real del DataLoader
images, labels = next(iter(train_loader))
logits = model(images)
print("Batch real   :", tuple(images.shape))
print("Salida       :", tuple(logits.shape))
print("Predicción (sin entrenar) de las 8 primeras:", logits.argmax(1)[:8].tolist())
print("Etiquetas reales                            :", labels[:8].tolist())

## 8. Entrenamiento

El entrenamiento completo se ejecuta desde la raíz del repositorio con:

```bash
python src/train.py
```

Al terminar, se muestran aquí los gráficos generados.

In [ ]:
from IPython.display import Image, display
out = Path.cwd().parent / "outputs"
for f in ["loss_accuracy.png", "confusion_matrix.png"]:
    if (out / f).exists():
        display(Image(filename=str(out / f)))
    else:
        print(f"{f} todavía no existe: ejecuta python src/train.py")